# Graph RAG Pipeline for KFlow

This notebook shows how to define and exercise the RAG pipeline from Python against the deployed stack. It covers direct API usage, LangChain retrieval, LlamaIndex/Ollama setup, and a LangGraph workflow that indexes, retrieves, and answers with context.

Default local endpoint: `https://localhost`. For production, set `BASE_URL=https://your-domain.example`.

In [ ]:
%pip install -U requests qdrant-client langchain langchain-ollama langchain-qdrant langgraph llama-index llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
import os
import requests

BASE_URL = os.getenv("BASE_URL", "https://localhost")
VERIFY_TLS = os.getenv("VERIFY_TLS", "false").lower() == "true"

def api(path, method="GET", **kwargs):
    response = requests.request(method, f"{BASE_URL}{path}", verify=VERIFY_TLS, timeout=60, **kwargs)
    response.raise_for_status()
    return response.json() if response.content else {}

api("/rag/health")

## Web UI Definition

In Directus (`/`), define collections such as `products`, `documents`, `assets`, and `rag_sources`. Store product facts, URLs, file references, SKU metadata, and semantic tags. Upload raw source files into CMS assets or place batch files under `data/raw/` on the server. The RAG API indexes text directly and records media files as searchable metadata.

## CLI/API Pipeline

Use the RAG API directly for CMS payloads and raw-file indexing. This is the same path used by shell tests and CI.

In [ ]:
payload = {
    "name": "sku-123",
    "source_type": "cms-product",
    "content": "SKU 123 is a semantic wiki webshop demo product with QR tracking, CMS metadata, and Graph RAG context.",
    "metadata": {"sku": "sku-123", "collection": "products", "tags": ["demo", "rag"]},
}
api("/rag/ingest", method="POST", json=payload)

In [ ]:
api("/rag/index/raw", method="POST")

In [ ]:
api("/rag/query", params={"q": "What is sku-123?", "top_k": 5})

In [ ]:
api("/rag/chat", method="POST", json={"message": "Summarize sku-123 for a webshop operator", "top_k": 5})

## LangChain Retrieval Against Qdrant

The deployed RAG API already writes vectors to Qdrant. Use LangChain when you want a client-side retriever or experimentation workflow. For production, prefer routing writes through `/rag/ingest` so metadata stays consistent.

In [ ]:
from langchain_core.embeddings import Embeddings
from qdrant_client import QdrantClient

class KFlowHashEmbeddings(Embeddings):
    def __init__(self, size=384):
        self.size = size

    def _embed(self, text):
        import hashlib, math, re
        vector = [0.0] * self.size
        for token in re.findall(r"[a-zA-Z0-9_]+", text.lower()):
            digest = hashlib.sha256(token.encode()).digest()
            idx = int.from_bytes(digest[:4], "big") % self.size
            vector[idx] += 1.0 if digest[4] % 2 == 0 else -1.0
        norm = math.sqrt(sum(v * v for v in vector)) or 1.0
        return [v / norm for v in vector]

    def embed_documents(self, texts):
        return [self._embed(text) for text in texts]

    def embed_query(self, text):
        return self._embed(text)

qdrant_url = os.getenv("QDRANT_URL", "http://localhost:6333")
collection = os.getenv("RAG_COLLECTION", "kflow_documents")
client = QdrantClient(url=qdrant_url)
embedding = KFlowHashEmbeddings()
client.get_collections()

If you run with Ollama enabled, switch to `langchain_ollama.OllamaEmbeddings` and a model such as `nomic-embed-text`; make sure the same embedding model is used for both indexing and querying.

In [ ]:
# Optional Ollama chat model through LangChain.
# Requires: docker compose --env-file .env --profile llm up -d ollama
from langchain_ollama import ChatOllama

ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
llm = ChatOllama(model=os.getenv("OLLAMA_MODEL", "phi3:mini"), base_url=ollama_base_url)
# llm.invoke("Answer in one sentence: what is Graph RAG?")

## LlamaIndex Ollama Setup

Use LlamaIndex when you want its readers, indexes, or query engines. Keep the VPS memory budget in mind; local Ollama models should run only with the optional `llm` profile or on a separate host.

In [ ]:
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

llamaindex_llm = Ollama(model=os.getenv("OLLAMA_MODEL", "phi3:mini"), base_url=ollama_base_url, request_timeout=120.0)
llamaindex_embed = OllamaEmbedding(model_name=os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text"), base_url=ollama_base_url)
# llamaindex_llm.complete("Explain how Directus content becomes RAG context.")

## LangGraph Orchestration

This graph defines a deterministic pipeline: ingest content, retrieve context, then answer using the deployed RAG chat endpoint.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class GraphRagState(TypedDict, total=False):
    name: str
    content: str
    question: str
    ingest: dict
    retrieval: dict
    answer: dict

def ingest_node(state: GraphRagState):
    result = api("/rag/ingest", method="POST", json={
        "name": state["name"],
        "content": state["content"],
        "source_type": "langgraph-notebook",
        "metadata": {"workflow": "graph-rag"},
    })
    return {"ingest": result}

def retrieve_node(state: GraphRagState):
    return {"retrieval": api("/rag/query", params={"q": state["question"], "top_k": 5})}

def answer_node(state: GraphRagState):
    return {"answer": api("/rag/chat", method="POST", json={"message": state["question"], "top_k": 5})}

graph = StateGraph(GraphRagState)
graph.add_node("ingest", ingest_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("answer", answer_node)
graph.set_entry_point("ingest")
graph.add_edge("ingest", "retrieve")
graph.add_edge("retrieve", "answer")
graph.add_edge("answer", END)
workflow = graph.compile()

workflow.invoke({
    "name": "graph-rag-demo",
    "content": "Graph RAG connects CMS entities, semantic metadata, vector retrieval, and chat answers.",
    "question": "How does Graph RAG help the webshop CMS?",
})